# Flood detection: Colab GPU training

**Restricted-data warning.** This project's training data includes the **Flood Master Database** (AIIA Lab, Aristotle University of Thessaloniki), licensed for non-commercial research use only, with **no redistribution**. `data/processed/` (built from it, among other sources) must stay on **team-controlled storage** -- the shared Google Drive folder this notebook mounts below -- and must never be committed to git, uploaded to a public bucket, or shared outside the team. Do not change the Drive mount cell to point at anyone's personal, non-team Drive.

Local CPU training is fast enough for this project's dataset size (see `docs/PLAN.md` section 1) that this notebook is optional -- a GPU option for whoever wants faster epochs, not a requirement.

What this notebook does:
1. Clones `floodmark` at the `ml` branch.
2. Installs `flood-ml/requirements-train.txt`.
3. Mounts the team Drive and points `data/processed` at the shared copy.
4. Runs `train.train` for Stage A and both Stage B variants (see `docs/PLAN.md` section 5 / `docs/phase_reports/phase3_modeling.md` for why there are two Stage B variants).

In [ ]:
!nvidia-smi || echo "no GPU visible -- training will fall back to CPU, which is fine, just slower"

In [ ]:
REPO_URL = "https://github.com/joshualamiy/floodmark.git"
BRANCH = "ml"

!test -d floodmark || git clone --branch {BRANCH} {REPO_URL}
%cd floodmark
!git checkout {BRANCH}
!git pull

In [ ]:
%cd flood-ml
!pip install -q -r requirements-train.txt

## Mount the team Drive

`DRIVE_PROCESSED_DIR` must point at the shared team folder that holds a copy of `data/processed/` (manifest, images, `VERSION`) -- ask in the team channel for the current path if you don't have it. This notebook only reads from it; it never writes restricted data back out anywhere else.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/floodmark_team/data_processed"  # team-controlled path -- update if needed

In [ ]:
import os

os.makedirs("data", exist_ok=True)
if not os.path.islink("data/processed") and not os.path.isdir("data/processed"):
    os.symlink(DRIVE_PROCESSED_DIR, "data/processed")
print(sorted(os.listdir("data/processed"))[:10])

## Train

Fast baseline first (MobileNetV3Small, head-only), then fine-tuning and EfficientNetB0 if the baseline looks reasonable. Every run appends a row to `reports/runs.csv` and writes TensorBoard logs to `logs/<run_id>/`.

In [ ]:
!PYTHONPATH=src python -m train.train --stage a --backbone mobilenetv3small \
    --epochs-head 10 --epochs-ft 6 --ft-layers 30 --notes "colab gpu run"

In [ ]:
!PYTHONPATH=src python -m train.train --stage b --variant spec --backbone mobilenetv3small \
    --epochs-head 10 --epochs-ft 6 --ft-layers 30 --notes "colab gpu run"

In [ ]:
!PYTHONPATH=src python -m train.train --stage b --variant mixed --backbone mobilenetv3small \
    --epochs-head 10 --epochs-ft 6 --ft-layers 30 --wet-upweight 8 --notes "colab gpu run"

## (Optional) EfficientNetB0

Slower per epoch on CPU (about 4x MobileNetV3Small), but a GPU runtime makes this cheap -- worth trying here even if the CPU-only baseline in `docs/phase_reports/phase3_modeling.md` didn't.

In [ ]:
!PYTHONPATH=src python -m train.train --stage a --backbone efficientnetb0 \
    --epochs-head 10 --epochs-ft 6 --ft-layers 40 --notes "colab gpu run"

## Pull results back down

`reports/runs.csv` is text and safe to copy anywhere (including back into git via a normal PR from a laptop). `models/` and `logs/` are gitignored and contain trained weights -- keep those on team-controlled storage only, same as the restricted training data above. Zip and copy them to the same Drive folder if you want to hand a checkpoint to the export step, which can run on CPU: `python -m train.export_onnx --run-id <run_id> --stage a --out models/stage_a.onnx`.

In [ ]:
import shutil

DRIVE_MODELS_DIR = "/content/drive/MyDrive/floodmark_team/models_colab"  # team-controlled path -- update if needed
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
if os.path.isdir("models"):
    shutil.copytree("models", DRIVE_MODELS_DIR, dirs_exist_ok=True)
print("copied models/ to", DRIVE_MODELS_DIR)